# 17 — Instrumental Variables for Business Questions
**References:** Angrist & Pischke (2009) *Mostly Harmless Econometrics*, Ch. 4 · Imbens & Angrist (1994, *Econometrica*) · Stock & Yogo (2005) "Testing for Weak Instruments in Linear IV Regression" · Staiger & Stock (1997, *Econometrica*)

**Prerequisites:** causal_inference_course/07_instrumental_variables.ipynb (full theory: LATE
theorem, complier/always-taker/never-taker typology, weak-instrument asymptotics — not
repeated here).
**Connects to:** causal_inference_course/07_instrumental_variables.ipynb (identification theory
this notebook applies); 12_regression_discontinuity_business.ipynb (fuzzy RD is IV at a
threshold — a related but distinct business design); 14_fixed_effects_panel.ipynb (fixed effects
handle time-invariant confounding but not simultaneity — the gap IV fills).

## Narrative thread
```
Business question: does raising price actually reduce demand, or is price set IN RESPONSE to demand?
   -> Simultaneity/reverse causality -> OLS is uninterpretable
   -> An instrument: a supply-cost shock that shifts price but not demand directly
   -> Relevance & exclusion, examined in the business context
   -> First-stage F-stat, Stock-Yogo critical values
   -> Full worked 2SLS example with linearmodels.IV2SLS
```

## Why this notebook exists

Prices, marketing spend, and many other business "treatments" are typically **chosen by the
business in response to demand or performance** — a manager raises price when demand is
strong, or increases ad spend when a product is underperforming. Regressing an outcome on such
an endogenous decision variable estimates a mix of the causal effect and the reverse
relationship, and fixed effects (`14_fixed_effects_panel.ipynb`) cannot fix this because the
problem isn't a stable omitted confounder — it's simultaneity. `causal_inference_course/07`
derives the full IV/LATE theory; this notebook works a concrete pricing-elasticity example end
to end and treats relevance and exclusion as *business* judgment calls, not just textbook
assumptions.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(17)

## Business setting: does price cause lower demand, or does demand drive price?

A regional grocery chain wants its true **price elasticity of demand** for a staple product,
to inform a pricing decision. But store managers routinely raise prices when they observe
strong local demand (limited supply, high foot traffic) and cut prices when demand is weak —
so price and demand are jointly determined, and a plain regression of quantity sold on price
is contaminated by this simultaneity as well as by unobserved local demand shocks.

**The instrument**: an upstream **supplier input-cost shock** that varies by week and region
(e.g., a regional fuel-surcharge or a temporary supplier shortage) that shifts the chain's
wholesale cost, and therefore the retail price it sets — but has no direct effect on
consumers' desire to buy the product, other than through the price it induces.

In [2]:
# ── Simulate weekly store-level price, quantity, an unobserved demand shock, and a cost instrument ──
n_stores = 80
n_weeks = 30
n = n_stores * n_weeks

store_id = np.repeat(np.arange(n_stores), n_weeks)
week_id = np.tile(np.arange(n_weeks), n_stores)

unobserved_demand_shock = np.random.normal(0, 1, n)          # drives BOTH price (managers react to it) and quantity
cost_shock = np.random.normal(0, 1, n)                        # the instrument: supplier input-cost shock

true_elasticity = -1.8   # true causal effect: 1-unit price increase -> 1.8-unit drop in log quantity

# price responds to the cost shock (relevance) AND to the unobserved demand shock (this is the endogeneity)
price = 5.0 + 0.9 * cost_shock + 0.7 * unobserved_demand_shock + np.random.normal(0, 0.3, n)

# quantity responds to price causally, AND directly to the same unobserved demand shock (confounding)
# crucially, quantity does NOT depend on cost_shock except through price (the exclusion restriction)
log_quantity = (
    4.0 + true_elasticity * price + 1.1 * unobserved_demand_shock + np.random.normal(0, 0.25, n)
)

df = pd.DataFrame({
    "store": store_id, "week": week_id, "price": price,
    "cost_shock": cost_shock, "log_quantity": log_quantity,
})
df.head()

,store,week,price,cost_shock,log_quantity
0,0,0,5.653550,0.607063,-5.676211
1,0,1,3.365722,-0.917860,-4.050507
2,0,2,5.028878,-0.262890,-4.256271
3,0,3,5.394921,-1.312953,-4.746595
4,0,4,6.391696,1.071152,-5.957772


In [3]:
# ── Naive OLS: biased toward zero (or even the wrong sign) by the demand-shock confound ──
ols = smf.ols("log_quantity ~ price", data=df).fit(cov_type="HC3")
print(f"Naive OLS elasticity estimate: {ols.params['price']:.3f}  (true = {true_elasticity})")
print("Biased toward zero: high-demand weeks have both higher prices AND higher quantity,")
print("masking the true negative causal relationship.")

Naive OLS elasticity estimate: -1.244  (true = -1.8)
Biased toward zero: high-demand weeks have both higher prices AND higher quantity,
masking the true negative causal relationship.


## Relevance and the exclusion restriction, examined in context

Two conditions must both hold for `cost_shock` to be a valid instrument (see
`causal_inference_course/07` for the formal LATE derivation):

- **Relevance** (testable): the instrument must actually move price. If suppliers pass costs
  through to retail price only weakly or slowly, the instrument is weak — a concrete business
  risk if, e.g., stores absorb cost shocks into margin rather than passing them to
  shoppers.
- **Exclusion** (not directly testable, an argument): the cost shock must affect quantity
  demanded **only through price** — not, say, through a supply *shortage* that also limits
  what's on the shelf (a stockout channel) or through a shared macro shock that independently
  affects both wholesale costs and local demand (e.g., a regional recession lowering both).
  This is a genuine business judgment call: is a fuel-surcharge shock plausibly unrelated to
  consumer demand for groceries, other than via the price it induces? Usually yes, but a
  careful analyst should rule out stockouts and correlated macro shocks explicitly before
  trusting it.

## First stage and its F-statistic

In [4]:
first_stage = smf.ols("price ~ cost_shock", data=df).fit(cov_type="HC3")
print(first_stage.summary().tables[1])
f_stat = first_stage.fvalue
print(f"\nFirst-stage F-statistic: {f_stat:.1f}")

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      5.0096      0.016    316.913      0.000       4.979       5.041
cost_shock     0.8867      0.015     57.931      0.000       0.857       0.917

First-stage F-statistic: 3356.0


## Weak-instrument diagnostics: Stock-Yogo critical values

Staiger & Stock (1997)'s well-known rule of thumb is **F > 10**; Stock & Yogo (2005) formalize
critical values that depend on how much finite-sample bias or size distortion one is willing to
tolerate. For a single endogenous regressor and single instrument (our case), Stock-Yogo's 10%
maximal-IV-size critical value is around **16.38**; the 2SLS 10%-worst-case relative-bias
critical value is around **16.38** as well for one instrument (the exact table value depends on
the number of instruments and endogenous regressors — see Stock & Yogo 2005, Table 1). Modern
work (Lee, McCrary, Moreira & Porter 2022) argues for **conventional inference to require F well
above 10**, sometimes suggesting F > 104.7 for a nominal 5% test to actually have close to 5%
size. Treat these as a spectrum of increasingly conservative thresholds, not a single bright
line.

In [5]:
thresholds = {"Staiger-Stock rule of thumb": 10, "Stock-Yogo 10% maximal IV size (1 instrument)": 16.38,
              "Lee et al. (2022) conservative threshold": 104.7}
print(f"Observed first-stage F: {f_stat:.1f}\n")
for name, thresh in thresholds.items():
    verdict = "PASSES" if f_stat > thresh else "fails"
    print(f"{name:48s} (>{thresh:>6}): {verdict}")

Observed first-stage F: 3356.0

Staiger-Stock rule of thumb                      (>    10): PASSES
Stock-Yogo 10% maximal IV size (1 instrument)    (> 16.38): PASSES
Lee et al. (2022) conservative threshold         (> 104.7): PASSES


## Full worked 2SLS example

Manual 2SLS SEs are wrong (`causal_inference_course/07`) — the correct covariance accounts for
the first-stage estimation uncertainty. We use `linearmodels.iv.IV2SLS`, which implements the
proper 2SLS/GMM covariance.

In [6]:
iv_model = IV2SLS.from_formula(
    "log_quantity ~ 1 + [price ~ cost_shock]", data=df
).fit(cov_type="robust")
print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:           log_quantity   R-squared:                      0.5502
Estimator:                    IV-2SLS   Adj. R-squared:                 0.5501
No. Observations:                2400   F-statistic:                    4980.8
Date:                Sun, Jul 05 2026   P-value (F-stat)                0.0000
Time:                        15:55:17   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      4.1919     0.1323     31.677     0.0000      3.9325      4.4512
price         -1.8329     0.0260    -70.575     0.00

In [7]:
print(f"2SLS elasticity estimate: {iv_model.params['price']:.3f}  (true = {true_elasticity})")
print(f"Naive OLS estimate for comparison:  {ols.params['price']:.3f}")
print(f"\nFirst-stage F-stat (from IV2SLS diagnostics): {iv_model.first_stage.diagnostics.loc['price', 'f.stat']:.1f}")

2SLS elasticity estimate: -1.833  (true = -1.8)
Naive OLS estimate for comparison:  -1.244

First-stage F-stat (from IV2SLS diagnostics): 3366.2


## A second business flavor: encouragement designs as an instrument

Beyond cost-shock instruments, a common business IV design is an **encouragement design**: a
randomized nudge (an email reminder, a push notification, a small discount code) that
*encourages* but does not force adoption of some behavior (e.g., enabling two-factor
authentication, switching to autopay). The randomized nudge itself is the instrument:
relevance holds by design (a randomized nudge changes take-up), and exclusion requires that the
nudge affects the outcome only through inducing the behavior, not directly (e.g., a well-worded
email shouldn't change engagement on its own, apart from getting people to adopt the feature).
This is the same LATE machinery, applied to a design a growth team can actually run
prospectively rather than having to find a naturally occurring cost shock after the fact — see
`causal_inference_course/07` for the compliance typology (compliers/always-takers/never-takers)
that governs what such a design identifies.

## Practitioner checklist

1. **State the exclusion restriction as a business argument, in plain language**, and try
   actively to falsify it (are there other channels from the instrument to the outcome?)
   before running any IV regression.
2. **Always report the first-stage F-statistic** and compare it against more than one
   threshold — passing "F > 10" alone is a weak bar by modern standards.
3. **Prefer a design-based instrument you understand end to end** (a genuine cost shock, a
   randomized encouragement nudge) over a "found" instrument with a plausible-sounding story
   but an opaque mechanism.
4. **Use the proper 2SLS covariance from a library** (`linearmodels.iv.IV2SLS`,
   `statsmodels`'s own IV tools) rather than manually running two OLS regressions and using the
   second stage's naive SEs.
5. **Remember what IV estimates**: with heterogeneous effects, 2SLS recovers a LATE for the
   subpopulation whose price (or behavior) responds to the instrument — not necessarily the
   ATE for all stores/customers (`causal_inference_course/07`).

## Key takeaways

| Concept | Statement |
|---|---|
| Simultaneity | Price and demand determine each other; OLS conflates cause and reverse-cause |
| Instrument | A supply-cost shock (or randomized nudge) that moves the endogenous variable but not the outcome directly |
| Relevance | Testable via the first-stage F-statistic |
| Exclusion | Not testable; a business argument that must be actively scrutinized |
| Weak instruments | F > 10 is a weak modern bar; Stock-Yogo and Lee et al. (2022) suggest much higher thresholds |
| 2SLS | Use a library's proper covariance, never manual two-stage OLS standard errors |

## References

| Author(s) | Title | Role here |
|---|---|---|
| Angrist & Pischke (2009) | *Mostly Harmless Econometrics*, Ch. 4 | IV/2SLS theory |
| Imbens & Angrist (1994, *Econometrica*) | "Identification and Estimation of Local Average Treatment Effects" | LATE theorem |
| Staiger & Stock (1997, *Econometrica*) | "Instrumental Variables Regression with Weak Instruments" | F > 10 rule of thumb |
| Stock & Yogo (2005) | "Testing for Weak Instruments in Linear IV Regression" | Formal weak-instrument critical values |
